MD# ADVANCE NATURAL LANGUAGE PROCESSING (AML23702) - Lab Exercise 3

**Name:** P Prithvi Raj Kandan &nbsp;|&nbsp; **USN:** 1GA23AI033 &nbsp;|&nbsp; Dept. of AI & ML, Global Academy of Technology

**Aim:** Build a simple Retrieval-Augmented Generation (RAG) system over the Neo4j Movies dataset: retrieve movie facts for a user query and generate a natural-language answer with an LLM.

MD**Setup.** The knowledge source is the official Neo4j *Movies* example graph (`:play movies`, saved as `../data/movies.cypher`).

* If `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` are set, the notebook queries the live Neo4j database with Cypher.
* Otherwise it reads `../data/movies.json`, an export of the same graph.
* If `LLM_API_KEY` is set (OpenAI-compatible endpoint, default Groq), an LLM writes the answer. **If it is not set, a template answer is built from the retrieved facts and labelled as a fallback.**


In [1]:
%pip install -q neo4j requests scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, requests
from pathlib import Path

DATA = Path("../data")
USE_NEO4J = bool(os.getenv("NEO4J_URI"))
if USE_NEO4J:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(os.getenv("NEO4J_URI"),
                                  auth=(os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "")))
    def cypher(query, **params):
        return [r.data() for r in driver.execute_query(query, **params).records]
    print("Knowledge source: live Neo4j at", os.getenv("NEO4J_URI"))
    print(cypher("MATCH (m:Movie) RETURN count(m) AS movies"),
          cypher("MATCH (p:Person) RETURN count(p) AS people"),
          cypher("MATCH ()-[r]->() RETURN count(r) AS relationships"))
else:
    from nlp import MovieGraph      # same dataset, read from ../data/movies.json
    g = MovieGraph()
    print("Knowledge source:", g.mode, "-", len(g.titles), "movies,", len(g.names), "people")


Knowledge source: bundled JSON (data/movies.json) - 38 movies, 133 people


In [3]:
# Retrieval step: Cypher queries for movies and people
def movie_facts(title):
    rows = cypher("""
        MATCH (m:Movie {title: $title})
        OPTIONAL MATCH (p:Person)-[r]->(m)
        RETURN m.released AS released, m.tagline AS tagline,
               p.name AS person, type(r) AS rel, r.roles AS roles""", title=title)
    facts = [f"{title} was released in {rows[0]['released']}.",
             f"Tagline: {rows[0]['tagline']}"]
    for rel, label in [("DIRECTED", "Directed by"), ("WROTE", "Written by"), ("PRODUCED", "Produced by")]:
        names = sorted(r["person"] for r in rows if r["rel"] == rel)
        if names:
            facts.append(f"{label}: {', '.join(names)}.")
    cast = sorted(f"{r['person']} as {', '.join(r['roles'])}" for r in rows if r["rel"] == "ACTED_IN")
    if cast:
        facts.append("Cast: " + "; ".join(cast) + ".")
    return facts

def person_facts(name):
    rows = cypher("""
        MATCH (p:Person {name: $name})-[r]->(m:Movie)
        RETURN p.born AS born, type(r) AS rel, m.title AS title, m.released AS year
        ORDER BY year""", name=name)
    facts = [f"{name} was born in {rows[0]['born']}."]
    for rel, verb in [("ACTED_IN", "acted in"), ("DIRECTED", "directed"), ("PRODUCED", "produced"), ("WROTE", "wrote")]:
        films = [f"{r['title']} ({r['year']})" for r in rows if r["rel"] == rel]
        if films:
            facts.append(f"{name} {verb}: {', '.join(films)}.")
    return facts

def movies_in_year(year):
    return [r["t"] for r in cypher("MATCH (m:Movie {released: $y}) RETURN m.title AS t ORDER BY t", y=year)]

if USE_NEO4J:
    titles = [r["t"] for r in cypher("MATCH (m:Movie) RETURN m.title AS t")]
    names = [r["n"] for r in cypher("MATCH (p:Person) RETURN p.name AS n")]
else:   # offline: same facts from the JSON export
    movie_facts, person_facts, movies_in_year = g.movie_facts, g.person_facts, g.movies_in_year
    titles, names = g.titles, g.names

def retrieve(question):
    q = question.lower()
    context = []
    for t in sorted(titles, key=len, reverse=True):
        if t.lower() in q:
            context += movie_facts(t)
            q = q.replace(t.lower(), " ")   # do not match a shorter title inside a longer one
    for n in names:
        if n.lower() in q:
            context += person_facts(n)
    for year in re.findall(r"\b(?:19|20)\d\d\b", question):
        films = movies_in_year(int(year))
        context.append(f"Movies released in {year}: {', '.join(films)}.")
    return context

for fact in retrieve("Who directed The Matrix and who played Neo?"):
    print("-", fact)

- The Matrix was released in 1999.
- Tagline of The Matrix: "Welcome to the Real World".
- Directed by: Lana Wachowski, Lilly Wachowski.
- Produced by: Joel Silver.
- Cast of The Matrix: Carrie-Anne Moss as Trinity; Emil Eifrem as Emil; Hugo Weaving as Agent Smith; Keanu Reeves as Neo; Laurence Fishburne as Morpheus.


In [4]:
# Generation step: LLM if a key is set, otherwise a labelled template answer
def generate(question, context):
    key = os.getenv("LLM_API_KEY")
    if key and context:
        base = os.getenv("LLM_BASE_URL", "https://api.groq.com/openai/v1")
        model = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")
        prompt = ("Answer using only these movie facts.\n" + "\n".join(context) +
                  f"\n\nQuestion: {question}\nAnswer:")
        r = requests.post(f"{base}/chat/completions", headers={"Authorization": f"Bearer {key}"},
                          json={"model": model, "messages": [{"role": "user", "content": prompt}]}, timeout=60)
        return f"[LLM: {model}] " + r.json()["choices"][0]["message"]["content"]
    if not context:
        return "[Template fallback] No matching movie or person was found in the graph."
    return "[Template fallback - no LLM_API_KEY set] From the Movies graph: " + " ".join(context[:6])

def rag(question):
    context = retrieve(question)
    print("Question:", question)
    print("Retrieved facts:", len(context))
    print("Answer:", generate(question, context), "\n")

print("LLM key set:", bool(os.getenv("LLM_API_KEY")))

LLM key set: False


In [5]:
questions = [ln.strip() for ln in (DATA / "questions.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
for question in questions:
    rag(question)


Question: Who directed The Matrix and who played Neo?
Retrieved facts: 5
Answer: [Template fallback - no LLM_API_KEY set] From the Movies graph: The Matrix was released in 1999. Tagline of The Matrix: "Welcome to the Real World". Directed by: Lana Wachowski, Lilly Wachowski. Produced by: Joel Silver. Cast of The Matrix: Carrie-Anne Moss as Trinity; Emil Eifrem as Emil; Hugo Weaving as Agent Smith; Keanu Reeves as Neo; Laurence Fishburne as Morpheus. 

Question: Which movies did Tom Hanks act in?
Retrieved facts: 3
Answer: [Template fallback - no LLM_API_KEY set] From the Movies graph: Tom Hanks was born in 1956. Tom Hanks acted in: A League of Their Own (1992), Apollo 13 (1995), Cast Away (2000), Charlie Wilson's War (2007), Cloud Atlas (2012), Joe Versus the Volcano (1990), Sleepless in Seattle (1993), That Thing You Do (1996), The Da Vinci Code (2006), The Green Mile (1999), The Polar Express (2004), You've Got Mail (1998). Tom Hanks directed: That Thing You Do (1996). 

Question: Wh

MD**Offline mode.** The Flask backend (`nlp.py`, class `MovieGraph`) uses the same retrieval logic on `../data/movies.json` when no `NEO4J_URI` is set, so the demo also works without a database server.
